# Cleaning_CYRV_products_dataset

### Conclusion

The dataset was inspected for missing values, duplicate records, duplicate product IDs, invalid product categories, negative and zero numeric values, fractional values, extreme values, and repeated product attributes.

No records were removed and no missing or unusual values were imputed because the identified anomalies could not be confirmed as data errors. Products with missing attributes, zero weight values, extreme measurements, and repeated product characteristics were therefore retained.

Numeric product attributes were converted from `float64` to nullable `Int64` after confirming that no fractional values were present. Original column names and category values were preserved to maintain consistency with the source dataset.

The cleaned dataset retains all 32,951 original product records and is ready for downstream analysis and integration with other datasets.

### Data Cleaning Summary

| Check | Result | Decision |
|---|---|---|
| Dataset shape | 32,951 rows × 9 columns | Preserved |
| Exact duplicate rows | 0 | No action required |
| Product ID uniqueness | 32,951 unique IDs; 0 duplicated IDs | Preserved |
| Missing descriptive attributes | 610 products | Retained; no reliable basis for imputation |
| Missing physical attributes | 2 products | Retained; no reliable basis for imputation |
| Missing both descriptive and physical attributes | 1 product | Retained to preserve `product_id` and potential referential integrity |
| Negative numeric values | 0 | No action required |
| Zero product weight | 4 products | Investigated and retained; insufficient evidence for correction |
| Zero product dimensions | 0 | No action required |
| Fractional numeric values | 0 | Numeric columns converted from `float64` to nullable `Int64` |
| Extreme numeric values | Present | Investigated and retained; insufficient evidence to classify as errors |
| Product categories | 73 non-null categories | Preserved; no formatting issues detected |
| Category formatting | 0 leading/trailing spaces, 0 uppercase values, 0 empty strings | No action required |
| Repeated product attributes | 1,158 rows | Retained because `product_id` values are unique |
| Column names | Original schema contains `lenght` | Preserved to maintain consistency with the source schema |
| Missing-value imputation | None performed | Missing values retained |
| Row removal | None performed | All 32,951 records retained |
| Saved-file validation | Shape, missing values, and duplicates match expected results | Validation passed |
| CSV dtype behavior | Nullable `Int64` columns reload as `float64` when missing values are present | Expected CSV behavior; no correction required |

**Output:** `CYRV_products_cleaned.csv`

## 1. Inspection

In [3]:
import pandas as pd

products = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_products_dataset.csv")

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [4]:
print("Shape:", products.shape)

print("\nData types:")
print(products.dtypes)

Shape: (32951, 9)

Data types:
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [5]:
missing_values = products.isna().sum()

missing_percent = (
    products.isna().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percent": missing_percent
}).sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
product_category_name,610,1.85
product_description_lenght,610,1.85
product_name_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_height_cm,2,0.01
product_length_cm,2,0.01
product_width_cm,2,0.01
product_id,0,0.00


In [8]:
products.duplicated().sum()
# Full duplicate rows: 0

np.int64(0)

In [9]:
print("Total rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())
print("Duplicated product IDs:", products["product_id"].duplicated().sum())

Total rows: 32951
Unique product IDs: 32951
Duplicated product IDs: 0


In [10]:
missing_610_cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

products[missing_610_cols].isna().value_counts()

product_category_name  product_name_lenght  product_description_lenght  product_photos_qty
False                  False                False                       False                 32341
True                   True                 True                        True                    610
Name: count, dtype: int64

In [11]:
products[
    products["product_category_name"].isna()
].head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [12]:
physical_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[
    products[physical_cols].isna().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
missing_description = products["product_category_name"].isna()

missing_physical = products[physical_cols].isna().any(axis=1)

print(
    "Missing description block:",
    missing_description.sum()
)

print(
    "Missing physical block:",
    missing_physical.sum()
)

print(
    "Missing both blocks:",
    (missing_description & missing_physical).sum()
)

Missing description block: 610
Missing physical block: 2
Missing both blocks: 1


## 2. Validation

In [14]:
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.0,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0


In [15]:
zero_weight = products[
    products["product_weight_g"].eq(0)
]

print("Products with weight = 0:", len(zero_weight))
display(zero_weight)

Products with weight = 0: 4


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


### Investigation of a suspicious value

We consider four instances where product_weight_g = 0 to be suspicious but leave them unchanged due to insufficient grounds for a correct replacement.We consider four instances where product_weight_g = 0 to be suspicious but leave them unchanged due to insufficient grounds for a correct replacement.


In [16]:
similar_products = products[
    (products["product_category_name"] == "cama_mesa_banho") &
    (products["product_length_cm"] == 30) &
    (products["product_height_cm"] == 25) &
    (products["product_width_cm"] == 30)
]

similar_products[
    [
        "product_id",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].sort_values("product_weight_g")

,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,53.0,528.0,1.0,0.0,30.0,25.0,30.0
1731,500909059ad72b36b2554150cc327edb,39.0,1140.0,1.0,3100.0,30.0,25.0,30.0


In [17]:
for column in numeric_cols:
    negative_count = (products[column] < 0).sum()
    print(f"{column}: {negative_count}")

product_name_lenght: 0
product_description_lenght: 0
product_photos_qty: 0
product_weight_g: 0
product_length_cm: 0
product_height_cm: 0
product_width_cm: 0


### Investigation of extreme values
We retain the upper extreme values ​​as potentially genuine outliers; there are no grounds for adjustment.

In [18]:
extreme_rows = pd.concat([
    products.nlargest(5, "product_weight_g"),
    products.nlargest(5, "product_length_cm"),
    products.nlargest(5, "product_height_cm"),
    products.nlargest(5, "product_width_cm"),
    products.nlargest(5, "product_photos_qty")
]).drop_duplicates()

extreme_rows.sort_index()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
88,7f34b85142d1ef9e077a0da9ada27385,moveis_decoracao,59.0,1750.0,3.0,2300.0,105.0,3.0,70.0
344,d0877f0094337c414d23f5a3c7bad20c,moveis_escritorio,48.0,489.0,2.0,30000.0,50.0,50.0,30.0
357,f2a1b32f85cad59ff2a8444154ac25f0,climatizacao,51.0,2959.0,4.0,7800.0,105.0,10.0,40.0
475,a2f4e28e50f60566eeb99f842ffc0fd9,instrumentos_musicais,57.0,374.0,1.0,5850.0,40.0,9.0,105.0
509,53f92b0474f91fcb5bd188c6a8075c38,utilidades_domesticas,54.0,2952.0,3.0,30000.0,76.0,51.0,51.0
788,11970aff9a8cf29a520127d0d8100901,moveis_decoracao,53.0,1750.0,3.0,2550.0,105.0,3.0,70.0
955,ceeba7d5636e59173cc5f484e913db3d,NaN,NaN,NaN,NaN,30000.0,65.0,65.0,65.0
967,f9aa001a859b11fd798bb386f3d07eb0,pet_shop,54.0,802.0,17.0,9000.0,33.0,30.0,44.0
1019,d19284bf0893e07e80c26fe4ad33458e,moveis_decoracao,47.0,1750.0,4.0,2550.0,105.0,3.0,70.0
1151,3c07c4a8d970b0ffad8a97bd6b5e478c,moveis_decoracao,56.0,1750.0,3.0,2450.0,105.0,3.0,70.0


### Category validation

In [22]:
print("Unique categories:", products["product_category_name"].nunique())

products["product_category_name"].value_counts(dropna=False)

Unique categories: 73


product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 74, dtype: int64

In [23]:
sorted(products["product_category_name"].dropna().unique())

['agro_industria_e_comercio',
 'alimentos',
 'alimentos_bebidas',
 'artes',
 'artes_e_artesanato',
 'artigos_de_festas',
 'artigos_de_natal',
 'audio',
 'automotivo',
 'bebes',
 'bebidas',
 'beleza_saude',
 'brinquedos',
 'cama_mesa_banho',
 'casa_conforto',
 'casa_conforto_2',
 'casa_construcao',
 'cds_dvds_musicais',
 'cine_foto',
 'climatizacao',
 'consoles_games',
 'construcao_ferramentas_construcao',
 'construcao_ferramentas_ferramentas',
 'construcao_ferramentas_iluminacao',
 'construcao_ferramentas_jardim',
 'construcao_ferramentas_seguranca',
 'cool_stuff',
 'dvds_blu_ray',
 'eletrodomesticos',
 'eletrodomesticos_2',
 'eletronicos',
 'eletroportateis',
 'esporte_lazer',
 'fashion_bolsas_e_acessorios',
 'fashion_calcados',
 'fashion_esporte',
 'fashion_roupa_feminina',
 'fashion_roupa_infanto_juvenil',
 'fashion_roupa_masculina',
 'fashion_underwear_e_moda_praia',
 'ferramentas_jardim',
 'flores',
 'fraldas_higiene',
 'industria_comercio_e_negocios',
 'informatica_acessorios',
 

In [24]:
category = products["product_category_name"]

print("Leading/trailing spaces:",
      category.dropna().ne(category.dropna().str.strip()).sum())

print("Contains uppercase:",
      category.dropna().str.contains(r"[A-Z]", regex=True).sum())

print("Empty/whitespace-only:",
      category.dropna().str.strip().eq("").sum())

Leading/trailing spaces: 0
Contains uppercase: 0
Empty/whitespace-only: 0


In [25]:
for col in numeric_cols:
    non_null = products[col].dropna()
    fractional_count = (non_null % 1 != 0).sum()
    print(f"{col}: {fractional_count}")

product_name_lenght: 0
product_description_lenght: 0
product_photos_qty: 0
product_weight_g: 0
product_length_cm: 0
product_height_cm: 0
product_width_cm: 0


In [26]:
physical_measurements = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in physical_measurements:
    print(f"{col}: {(products[col] == 0).sum()}")

product_weight_g: 4
product_length_cm: 0
product_height_cm: 0
product_width_cm: 0


### Attribute duplicates
1,158 rows investigated. Retained because product_id values are unique and identical metadata does not establish duplicate products.

In [27]:
product_attributes = products.columns.drop("product_id")

duplicate_attributes = products.duplicated(
    subset=product_attributes,
    keep=False
)

print(
    "Rows with duplicated product attributes:",
    duplicate_attributes.sum()
)

Rows with duplicated product attributes: 1158


In [28]:
products[
    duplicate_attributes
].sort_values(
    list(product_attributes)
).head(20)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
4205,0152cb427657428c06633fce6d721da4,alimentos,57.0,606.0,3.0,150.0,22.0,4.0,17.0
18510,1e114096c8024159923dcf2c857e4ca4,alimentos,57.0,606.0,3.0,150.0,22.0,4.0,17.0
13382,733162823b9a1dbc958b0988d32229da,artigos_de_festas,46.0,459.0,1.0,1350.0,23.0,17.0,18.0
19487,cc6a0d67ea3d63acca23c81500670843,artigos_de_festas,46.0,459.0,1.0,1350.0,23.0,17.0,18.0
14545,f046f5847e49bd22fcb24ecc59572165,automotivo,50.0,701.0,1.0,100.0,32.0,2.0,24.0
23770,0b9eab47f340cb0354b04f84b95940f9,automotivo,50.0,701.0,1.0,100.0,32.0,2.0,24.0
31164,7a0686c0403abe1a703aaaa9ecde289b,automotivo,50.0,701.0,1.0,100.0,32.0,2.0,24.0
2442,81b454070eecf89b21503cd5d313aa57,automotivo,50.0,2304.0,4.0,250.0,16.0,6.0,15.0
18159,4beaedeb352141968eb9374cae900996,automotivo,50.0,2304.0,4.0,250.0,16.0,6.0,15.0
19414,8c8361199b662437db44cff3bb7686ec,automotivo,56.0,1066.0,4.0,100.0,24.0,2.0,17.0


## Cleaning Decisions

Based on the inspection, validation, and investigation performed above, the following cleaning decisions were made:

* **Duplicate rows:** No exact duplicate rows were found. No rows were removed.
* **Product IDs:** All 32,951 `product_id` values are unique. No duplicate product identifiers were found.
* **Missing descriptive data:** 610 products have missing values in `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`. These records are retained because they still contain valid `product_id` values and may contain physical product information useful for joins and downstream analysis.
* **Missing physical data:** 2 products have missing values in all four physical attributes (`product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm`). These records are retained because there is insufficient evidence to reconstruct the missing values reliably.
* **Completely missing product metadata:** 1 product contains only a valid `product_id`, with both descriptive and physical attributes missing. The record is retained to preserve referential integrity for potential joins with other datasets.
* **Zero product weight:** 4 products have `product_weight_g = 0`. Investigation showed that all four belong to the same category and share the same dimensions, but there is insufficient evidence to determine their true weight. These values are therefore retained rather than imputed.
* **Negative and zero dimensions:** No negative numeric values and no zero values in product length, height, or width were found.
* **Extreme values:** Extreme values in weight, dimensions, and photo quantity were investigated. No sufficient evidence was found to classify them as data-entry errors, so they are retained.
* **Product categories:** 73 non-null product categories were found. No leading/trailing spaces, uppercase values, or empty category strings were detected. Category values are retained without modification.
* **Repeated product attributes:** 1,158 rows share the same available product attributes with at least one other row. They are retained because each record has a unique `product_id`, and identical metadata alone is insufficient to establish that the products are duplicates.
* **Numeric data types:** No fractional values were found in the numeric columns. Conversion from `float64` to nullable integer (`Int64`) can therefore be considered while preserving missing values.
* **Column names:** The source columns `product_name_lenght` and `product_description_lenght` contain the original spelling `lenght`. Any renaming should be treated as a schema decision rather than a correction to the underlying observations.


In [30]:
# Convert numeric product attributes to nullable integer type

integer_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[integer_cols] = products[integer_cols].astype("Int64")

In [31]:
products.dtypes

product_id                      str
product_category_name           str
product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
product_weight_g              Int64
product_length_cm             Int64
product_height_cm             Int64
product_width_cm              Int64
dtype: object

In [32]:
print("Shape:", products.shape)

print("\nMissing values:")
print(products.isna().sum())

print("\nDuplicate rows:", products.duplicated().sum())

Shape: (32951, 9)

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Duplicate rows: 0


In [33]:
print("Unique product IDs:", products["product_id"].nunique())
print("Duplicated product IDs:", products["product_id"].duplicated().sum())

print("\nKnown anomalies:")
print("Products with weight = 0:", (products["product_weight_g"] == 0).sum())

print(
    "Products missing description block:",
    products["product_category_name"].isna().sum()
)

print(
    "Products missing physical block:",
    products[physical_cols].isna().any(axis=1).sum()
)

Unique product IDs: 32951
Duplicated product IDs: 0

Known anomalies:
Products with weight = 0: 4
Products missing description block: 610
Products missing physical block: 2


In [34]:
output_path = "../CYRV_products_cleaned.csv"

products.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: ../CYRV_products_cleaned.csv


In [35]:
products_check = pd.read_csv(output_path)

print("Shape:", products_check.shape)

print("\nMissing values:")
print(products_check.isna().sum())

print("\nDuplicate rows:", products_check.duplicated().sum())

print("\nData types:")
print(products_check.dtypes)

Shape: (32951, 9)

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Duplicate rows: 0

Data types:
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
